# 📈 Prédiction du Prix des Actions avec un LSTM (PyTorch)

## Présentation du projet

Dans ce notebook, nous allons construire un modèle de **deep learning** capable de prédire le **prix de clôture du lendemain** d'une action boursière, en se basant sur l'historique des prix.

### 🎯 Ce que vous allez apprendre :
- Prétraiter des données de séries temporelles financières
- Créer un `Dataset` et un `DataLoader` PyTorch personnalisés
- Construire un modèle **LSTM** avec **PyTorch** (`nn.Module`)
- Entraîner le modèle avec des boucles train/validation
- Évaluer les performances avec le score **R²**

### 📊 Le Dataset
On utilise le **Stock Market Dataset** de Kaggle, qui contient l'historique de milliers d'actions (OHLCV : Open, High, Low, Close, Volume).

---
> ⚠️ **GPU recommandé** : `Runtime > Change runtime type > Hardware accelerator > GPU (T4)`

---
## 🔧 ÉTAPE 1 — Installation et import des bibliothèques

On installe les bibliothèques nécessaires et on vérifie que PyTorch détecte bien le GPU.

In [ ]:
# Installation des bibliothèques (si pas déjà présentes dans Colab)
!pip install -q kaggle scikit-learn torch

In [ ]:
# ── Imports standards ─────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pickle                        # Pour sauvegarder le scaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

# ── Configuration ─────────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid')
%matplotlib inline

# Détection automatique du GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Bibliothèques importées !")
print(f"🖥️  Appareil utilisé : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU : {torch.cuda.get_device_name(0)}")

---
## 📥 ÉTAPE 2 — Chargement et prétraitement des données

### Option A : Téléchargement via l'API Kaggle
### Option B : Upload manuel du fichier CSV

Nous allons travailler avec l'action **Apple (AAPL)** comme exemple, mais le code fonctionne avec n'importe quelle action du dataset.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION A : Téléchargement via Kaggle API
# ══════════════════════════════════════════════════════════════════════════════
# 1. Allez sur https://www.kaggle.com/settings > API > Create New Token
# 2. Cela télécharge un fichier 'kaggle.json'
# 3. Uploadez-le dans Colab avec la cellule ci-dessous

# from google.colab import files
# uploaded = files.upload()  # Sélectionnez votre kaggle.json

# # Placer le fichier au bon endroit
# os.makedirs('/root/.config/kaggle', exist_ok=True)
# !cp kaggle.json /root/.config/kaggle/
# !chmod 600 /root/.config/kaggle/kaggle.json

# # Télécharger le dataset
# !kaggle datasets download -d jacksoncrow/stock-market-dataset
# !unzip -q stock-market-dataset.zip -d stock_data

# ══════════════════════════════════════════════════════════════════════════════
# OPTION B : Upload manuel (plus simple)
# ══════════════════════════════════════════════════════════════════════════════
# Téléchargez AAPL.csv depuis Kaggle et uploadez-le directement ici :

# from google.colab import files
# uploaded = files.upload()  # Uploadez AAPL.csv

# ══════════════════════════════════════════════════════════════════════════════
# OPTION C (démo) : Génération de données synthétiques réalistes
# ══════════════════════════════════════════════════════════════════════════════
# Si vous n'avez pas accès au dataset, cette cellule génère des données
# qui reproduisent la structure du dataset Kaggle.

print("💡 Génération de données synthétiques pour la démo...")
np.random.seed(42)
n = 2520  # ~10 ans de trading (252 jours/an)
dates = pd.date_range('2013-01-01', periods=n, freq='B')  # 'B' = jours ouvrés

# Simulation d'un prix avec drift et volatilité (mouvement brownien)
returns = np.random.normal(0.0003, 0.015, n)
close   = 150 * np.cumprod(1 + returns)
open_   = close * (1 + np.random.normal(0, 0.005, n))
high    = np.maximum(close, open_) * (1 + np.abs(np.random.normal(0, 0.007, n)))
low     = np.minimum(close, open_) * (1 - np.abs(np.random.normal(0, 0.007, n)))
volume  = np.random.randint(50_000_000, 200_000_000, n).astype(float)

df_raw = pd.DataFrame({
    'Date':   dates,
    'Open':   open_,
    'High':   high,
    'Low':    low,
    'Close':  close,
    'Volume': volume,
    'Name':   'AAPL'
})

print(f"✅ Dataset généré : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")
df_raw.head()

In [ ]:
# ── Si vous utilisez le vrai dataset Kaggle, décommentez ces lignes ───────────
# Le fichier AAPL.csv est dans le dossier stocks/ du dataset

# df_raw = pd.read_csv('stock_data/stocks/AAPL.csv')
# df_raw['Date'] = pd.to_datetime(df_raw['Date'])
# print(f"✅ Dataset chargé : {df_raw.shape}")
# df_raw.head()

In [ ]:
# ── Prétraitement ─────────────────────────────────────────────────────────────

df = df_raw.copy()

# 1. Trier par date (important pour les séries temporelles !)
df = df.sort_values('Date').reset_index(drop=True)

# 2. Supprimer les colonnes inutiles
#    'Name' est une chaîne de texte, elle ne sert pas au modèle
df.drop(columns=['Name'], errors='ignore', inplace=True)

# 3. Créer la colonne cible : prix de clôture du LENDEMAIN
#    .shift(-1) décale les valeurs d'une ligne vers le haut
#    → la ligne i contient le Close du jour i+1
df['Target'] = df['Close'].shift(-1)

# 4. Supprimer la dernière ligne (pas de lendemain connu)
df.dropna(inplace=True)

print("📋 Aperçu du dataset prétraité :")
print(f"   Forme : {df.shape}")
print(f"   Colonnes : {list(df.columns)}")
df.head()

In [ ]:
# ── Visualisation du prix historique ─────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Prix de clôture
axes[0].plot(df['Date'], df['Close'], color='royalblue', linewidth=1)
axes[0].set_title('Prix de clôture historique (AAPL)', fontsize=13)
axes[0].set_ylabel('Prix ($)')

# Volume
axes[1].bar(df['Date'], df['Volume'], color='steelblue', alpha=0.5, width=1)
axes[1].set_title('Volume de transactions', fontsize=13)
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

plt.tight_layout()
plt.show()

In [ ]:
# ── Normalisation avec MinMaxScaler ──────────────────────────────────────────
# Pourquoi normaliser ?
#   Les prix (ex: 150$) et les volumes (ex: 100M) ont des échelles très différentes.
#   La normalisation ramène tout entre 0 et 1, ce qui aide le réseau à converger.

# Colonnes features (entrées) et target (sortie)
FEATURE_COLS = ['Open', 'High', 'Low', 'Close', 'Volume']
TARGET_COL   = 'Target'

# Scaler pour les features
scaler_X = MinMaxScaler()
# Scaler séparé pour la target → nécessaire pour inverser la normalisation
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(df[FEATURE_COLS].values)
y_scaled = scaler_y.fit_transform(df[[TARGET_COL]].values)  # double [[]] → shape (n,1)

print(f"✅ Données normalisées")
print(f"   X : shape {X_scaled.shape}, min={X_scaled.min():.2f}, max={X_scaled.max():.2f}")
print(f"   y : shape {y_scaled.shape}, min={y_scaled.min():.2f}, max={y_scaled.max():.2f}")

# Sauvegarde du scaler (utile pour de futures prédictions)
with open('scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)
print("💾 scaler_y.pkl sauvegardé !")

---
## 🗂️ ÉTAPE 3 — Préparation du Dataset PyTorch

PyTorch n'accepte pas directement des tableaux numpy. On doit :
1. **Créer des séquences** (fenêtre glissante) : le modèle voit N jours passés pour prédire le suivant
2. **Créer une classe `Dataset`** : interface standard de PyTorch pour les données
3. **Créer des `DataLoader`** : itérateurs qui servent les mini-batches pendant l'entraînement

In [ ]:
# ── Découpage chronologique Train / Validation / Test ─────────────────────────
# Convention standard pour les séries temporelles :
#   70% train | 15% validation | 15% test
# On ne mélange JAMAIS (pas de shuffle) — le futur ne peut pas précéder le passé !

n = len(X_scaled)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train_raw = X_scaled[:train_end]
y_train_raw = y_scaled[:train_end]

X_val_raw   = X_scaled[train_end:val_end]
y_val_raw   = y_scaled[train_end:val_end]

X_test_raw  = X_scaled[val_end:]
y_test_raw  = y_scaled[val_end:]

print("📐 Répartition des données :")
print(f"   Train      : {len(X_train_raw):>5} jours ({70}%)")
print(f"   Validation : {len(X_val_raw):>5} jours ({15}%)")
print(f"   Test       : {len(X_test_raw):>5} jours ({15}%)")

In [ ]:
# ── Classe Dataset personnalisée ──────────────────────────────────────────────
# En PyTorch, on hérite de torch.utils.data.Dataset et on implémente :
#   __len__     → retourne le nombre d'échantillons disponibles
#   __getitem__ → retourne le i-ème échantillon (X_i, y_i)

class StockDataset(Dataset):
    """
    Dataset de séquences pour la prédiction de prix d'actions.
    
    Chaque échantillon est une fenêtre de 'seq_len' jours (X)
    associée au prix cible du lendemain (y).
    """
    def __init__(self, X: np.ndarray, y: np.ndarray, seq_len: int = 30):
        """
        Args:
            X       : features normalisées, shape (n_jours, n_features)
            y       : target normalisée, shape (n_jours, 1)
            seq_len : taille de la fenêtre temporelle (nombre de jours passés)
        """
        self.seq_len = seq_len
        # Conversion en tenseurs PyTorch float32
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        # On perd 'seq_len' premiers indices (pas assez d'historique)
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        # X_seq : fenêtre de seq_len jours → shape (seq_len, n_features)
        X_seq = self.X[idx : idx + self.seq_len]
        # y_val : prix cible du jour suivant la fenêtre → scalaire
        y_val = self.y[idx + self.seq_len]
        return X_seq, y_val


# ── Instanciation des datasets ────────────────────────────────────────────────
SEQ_LEN = 30  # Fenêtre de 30 jours (un mois de trading)

train_dataset = StockDataset(X_train_raw, y_train_raw, SEQ_LEN)
val_dataset   = StockDataset(X_val_raw,   y_val_raw,   SEQ_LEN)
test_dataset  = StockDataset(X_test_raw,  y_test_raw,  SEQ_LEN)

print(f"✅ Datasets créés :")
print(f"   Train      : {len(train_dataset)} séquences")
print(f"   Validation : {len(val_dataset)} séquences")
print(f"   Test       : {len(test_dataset)} séquences")

# Vérification : forme d'un échantillon
x0, y0 = train_dataset[0]
print(f"\n📦 Forme d'un échantillon : X={x0.shape}, y={y0.shape}")

In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
# Le DataLoader découpe le dataset en mini-batches et les sert à l'entraînement.
#
# batch_size=32 → le modèle voit 32 séquences à la fois avant de mettre à jour
# shuffle=True  → mélange les batches en train (mais PAS les séquences elles-mêmes)
# num_workers=0 → threads de chargement (0 = thread principal, compatible Colab)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✅ DataLoaders prêts !")
print(f"   Train  : {len(train_loader)} batches × {BATCH_SIZE} = ~{len(train_loader)*BATCH_SIZE} échantillons")
print(f"   Val    : {len(val_loader)} batches")
print(f"   Test   : {len(test_loader)} batches")

---
## 🧠 ÉTAPE 4 — Définition du modèle LSTM

### Architecture PyTorch avec `nn.Module`

En PyTorch, tout modèle hérite de `nn.Module` et doit implémenter :
- `__init__` : déclare les couches
- `forward`  : définit comment les données traversent le réseau

### Architecture choisie :
```
Input → (batch, seq_len=30, features=5)
          ↓
    LSTM (128 unités, 2 couches)
    + Dropout(0.2) intégré
          ↓
    Dropout(0.2) supplémentaire
          ↓
    Linear(128 → 64)
          ↓
    ReLU
          ↓
    Linear(64 → 1)  ← prédiction du prix
```

In [ ]:
class StockLSTM(nn.Module):
    """
    Modèle LSTM pour la prédiction de prix d'actions.
    
    Architecture :
        LSTM (2 couches, 128 unités) → Dropout → Dense(64) → ReLU → Dense(1)
    """
    def __init__(self, input_size: int, hidden_size: int = 128,
                 num_layers: int = 2, dropout: float = 0.2):
        """
        Args:
            input_size  : nombre de features (colonnes) en entrée
            hidden_size : nombre de neurones dans chaque couche LSTM
            num_layers  : nombre de couches LSTM empilées
            dropout     : taux de dropout (régularisation)
        """
        super(StockLSTM, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # ── Couche LSTM ────────────────────────────────────────────────────────
        # batch_first=True → format entrée (batch, seq, features) au lieu de
        #                    (seq, batch, features) — plus intuitif
        # dropout → appliqué ENTRE les couches (pas sur la dernière)
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )

        # ── Dropout supplémentaire après LSTM ──────────────────────────────────
        self.dropout = nn.Dropout(dropout)

        # ── Couches fully connected ────────────────────────────────────────────
        self.fc1  = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(64, 1)   # Sortie = 1 valeur (prix prédit)

    def forward(self, x):
        """
        Passe avant (forward pass).
        
        Args:
            x : tenseur d'entrée, shape (batch_size, seq_len, input_size)
        Returns:
            out : prédiction, shape (batch_size, 1)
        """
        batch_size = x.size(0)

        # Initialisation des états cachés à zéro
        # h0 = état caché (hidden state), c0 = état cellulaire (cell state)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)

        # Passage dans le LSTM
        # lstm_out : (batch, seq_len, hidden_size) — sortie à chaque pas de temps
        # (hn, cn) : états finaux
        lstm_out, (hn, cn) = self.lstm(x, (h0, c0))

        # On prend UNIQUEMENT la sortie du DERNIER pas de temps
        # Cela correspond au résumé de toute la séquence
        last_output = lstm_out[:, -1, :]   # shape : (batch, hidden_size)

        # Couches denses
        out = self.dropout(last_output)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)               # shape : (batch, 1)

        return out


# ── Instanciation et envoi sur GPU ────────────────────────────────────────────
N_FEATURES = len(FEATURE_COLS)  # 5 : Open, High, Low, Close, Volume

model = StockLSTM(
    input_size  = N_FEATURES,
    hidden_size = 128,
    num_layers  = 2,
    dropout     = 0.2
).to(DEVICE)

print(model)
print(f"\n📊 Paramètres totaux : {sum(p.numel() for p in model.parameters()):,}")

---
## 🚀 ÉTAPE 5 — Entraînement du modèle

### Boucle d'entraînement PyTorch

Contrairement à Keras (`.fit()`), PyTorch nécessite d'écrire la boucle manuellement.  
C'est plus verbeux, mais bien plus flexible. Voici la structure à chaque epoch :

```
Pour chaque batch :
  1. optimizer.zero_grad()  → réinitialise les gradients
  2. output = model(X)      → passage avant (forward)
  3. loss = criterion(output, y)  → calcul de la loss
  4. loss.backward()        → rétropropagation (calcul des gradients)
  5. optimizer.step()       → mise à jour des poids
```

In [ ]:
# ── Configuration de l'entraînement ──────────────────────────────────────────

# Fonction de perte : MSE (Mean Squared Error) — standard pour la régression
criterion = nn.MSELoss()

# Optimiseur : Adam avec learning rate 0.001
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# Learning rate scheduler : réduit le LR si la val_loss stagne
# Patience=5 → si pas d'amélioration après 5 epochs, LR × 0.5
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

NUM_EPOCHS = 60

print(f"⚙️  Optimiseur  : Adam (lr=1e-3)")
print(f"⚙️  Loss        : MSELoss")
print(f"⚙️  Epochs max  : {NUM_EPOCHS}")
print(f"⚙️  Batch size  : {BATCH_SIZE}")

In [ ]:
# ── Fonctions utilitaires : une epoch d'entraînement et de validation ─────────

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Entraîne le modèle sur tous les batches du loader."""
    model.train()          # Mode entraînement (active dropout, etc.)
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()              # 1. Réinitialise les gradients
        predictions = model(X_batch)       # 2. Forward pass
        loss = criterion(predictions, y_batch)  # 3. Calcul de la loss
        loss.backward()                    # 4. Rétropropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Stabilité
        optimizer.step()                   # 5. Mise à jour des poids

        total_loss += loss.item() * len(X_batch)

    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    """Évalue le modèle sans calculer de gradients (plus rapide)."""
    model.eval()           # Mode évaluation (désactive dropout)
    total_loss = 0.0

    with torch.no_grad():  # Pas de calcul de gradient → économise la mémoire
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            total_loss += loss.item() * len(X_batch)

    return total_loss / len(loader.dataset)


print("✅ Fonctions d'entraînement définies.")

In [ ]:
# ── Boucle principale d'entraînement ─────────────────────────────────────────

train_losses = []
val_losses   = []
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 15  # Arrêt si pas d'amélioration après 15 epochs

print(f"🏋️  Démarrage de l'entraînement sur {DEVICE}...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss   = evaluate(model, val_loader, criterion, DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Scheduler : ajuste le LR si nécessaire
    scheduler.step(val_loss)

    # Sauvegarde du meilleur modèle
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
        best_marker = " ✅ (meilleur)"
    else:
        patience_counter += 1
        best_marker = ""

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}/{NUM_EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}{best_marker}")

    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\n⏹️  Early stopping à l'epoch {epoch} (pas d'amélioration depuis {EARLY_STOP_PATIENCE} epochs)")
        break

print(f"\n✅ Entraînement terminé ! Meilleure val_loss : {best_val_loss:.6f}")

In [ ]:
# ── Courbe d'apprentissage ────────────────────────────────────────────────────
# Ce graphique permet de diagnostiquer l'entraînement :
#   • Les deux courbes descendent ensemble → bon apprentissage
#   • val_loss remonte alors que train_loss descend → overfitting
#   • Les deux restent hautes → underfitting (modèle trop simple)

fig, ax = plt.subplots(figsize=(12, 5))

epochs_range = range(1, len(train_losses) + 1)
ax.plot(epochs_range, train_losses, label='Train Loss',      color='royalblue', linewidth=2)
ax.plot(epochs_range, val_losses,   label='Validation Loss', color='tomato',    linewidth=2, linestyle='--')

# Marquer l'epoch du meilleur modèle
best_epoch = val_losses.index(min(val_losses)) + 1
ax.axvline(best_epoch, color='green', linestyle=':', linewidth=1.5, label=f'Meilleur modèle (epoch {best_epoch})')

ax.set_title("Évolution de la Loss — Train vs Validation", fontsize=14)
ax.set_xlabel("Epochs")
ax.set_ylabel("MSE Loss")
ax.legend()
plt.tight_layout()
plt.show()

---
## 📏 ÉTAPE 6 — Évaluation du modèle

### Métriques utilisées

| Métrique | Formule | Interprétation |
|----------|---------|----------------|
| **MSE** | mean((y - ŷ)²) | Erreur quadratique moyenne |
| **RMSE** | √MSE | En mêmes unités que le prix ($) |
| **MAE** | mean(|y - ŷ|) | Erreur absolue moyenne ($) |
| **R²** | 1 - SS_res/SS_tot | 1 = parfait, 0 = modèle nul, <0 = mauvais |

> ⭐ Le **R²** est la métrique principale demandée dans l'exercice.

In [ ]:
# ── Chargement du meilleur modèle sauvegardé ─────────────────────────────────
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model.eval()
print("✅ Meilleur modèle chargé.")

# ── Prédictions sur le jeu de test ───────────────────────────────────────────
all_preds = []
all_true  = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        preds = model(X_batch).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y_batch.numpy())

all_preds = np.array(all_preds).reshape(-1, 1)
all_true  = np.array(all_true).reshape(-1, 1)

# ── Dé-normalisation : on revient aux valeurs en dollars ─────────────────────
preds_real = scaler_y.inverse_transform(all_preds).flatten()
true_real  = scaler_y.inverse_transform(all_true).flatten()

# ── Calcul des métriques ──────────────────────────────────────────────────────
mse  = np.mean((true_real - preds_real) ** 2)
rmse = np.sqrt(mse)
mae  = np.mean(np.abs(true_real - preds_real))
r2   = r2_score(true_real, preds_real)

print("\n📊 Performances sur le jeu de TEST (valeurs réelles) :")
print(f"   MSE  : {mse:.4f} $²")
print(f"   RMSE : {rmse:.4f} $  ← erreur moyenne en dollars")
print(f"   MAE  : {mae:.4f} $")
print(f"   R²   : {r2:.4f}    ← plus proche de 1 = meilleur")

In [ ]:
# ── Visualisation : Prédictions vs Valeurs réelles ────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# ── Graphique 1 : Série temporelle ────────────────────────────────────────────
axes[0].plot(true_real,  label='Prix réels',       color='steelblue', linewidth=1.5)
axes[0].plot(preds_real, label='Prédictions LSTM', color='tomato',    linewidth=1.5, linestyle='--')
axes[0].set_title(f'Prédiction du prix de clôture (R² = {r2:.4f})', fontsize=13)
axes[0].set_ylabel('Prix ($)')
axes[0].set_xlabel('Jours (jeu de test)')
axes[0].legend()

# ── Graphique 2 : Scatter plot (idéalement sur la diagonale) ─────────────────
axes[1].scatter(true_real, preds_real, alpha=0.4, s=10, color='royalblue')
min_val = min(true_real.min(), preds_real.min())
max_val = max(true_real.max(), preds_real.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Prédiction parfaite')
axes[1].set_title('Valeurs réelles vs Prédictions (scatter)', fontsize=13)
axes[1].set_xlabel('Valeurs réelles ($)')
axes[1].set_ylabel('Prédictions ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

print("💡 Plus les points sont proches de la ligne rouge, meilleures sont les prédictions.")

In [ ]:
# ── Sauvegarde finale du modèle et du scaler ──────────────────────────────────
# On sauvegarde tout ce qu'il faut pour faire de futures prédictions

# Modèle PyTorch
torch.save(model.state_dict(), 'stock_lstm_final.pth')

# Scalers (pour normaliser/dénormaliser les nouvelles données)
with open('scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)
with open('scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)

print("💾 Fichiers sauvegardés :")
print("   • stock_lstm_final.pth  → poids du modèle LSTM")
print("   • scaler_X.pkl          → scaler des features")
print("   • scaler_y.pkl          → scaler de la target")
print()
print("📋 Pour recharger le modèle plus tard :")
print("   model = StockLSTM(input_size=5)")
print("   model.load_state_dict(torch.load('stock_lstm_final.pth'))")

---
## 🎓 Conclusion

Vous avez construit un pipeline complet de prédiction de prix d'actions avec PyTorch :

| Étape | Ce qu'on a fait |
|-------|----------------|
| **Données** | Chargé et prétraité un dataset boursier OHLCV |
| **Target** | Créé la colonne « prix du lendemain » avec `.shift(-1)` |
| **Normalisation** | Appliqué `MinMaxScaler` séparément sur X et y |
| **Dataset** | Créé une classe `StockDataset` avec fenêtre glissante (30 jours) |
| **DataLoader** | Configuré les itérateurs train/val/test |
| **Modèle** | Défini un `StockLSTM` avec 2 couches LSTM + Dropout + Dense |
| **Entraînement** | Implémenté la boucle PyTorch (zero_grad → forward → loss → backward → step) |
| **Évaluation** | Calculé MSE, RMSE, MAE, R² sur les données dé-normalisées |
| **Sauvegarde** | Persisté le modèle (`.pth`) et les scalers (`.pkl`) |

### 🚀 Pour aller plus loin :
- Ajouter des **indicateurs techniques** : RSI, MACD, Bollinger Bands
- Tester avec d'autres actions du dataset Kaggle
- Essayer un **Transformer** (architecture attention) pour de meilleures performances
- Mettre en place une **stratégie de trading** basée sur les prédictions